%% [markdown]<br>
# Word Embedding for Sequence Processing<br>
**The goal of this practical is to use pre-trained word embedding for adressing the sequence prediction tasks studied in week 2: PoS and chunking.**

%%

In [ ]:
import numpy as np
import gensim.downloader as api
from gensim.models import KeyedVectors

%% [markdown]<br>
## 0) Loading PoS (or chunking) datasets (small or large)

%%

In [ ]:
def load(filename):
    listeDoc = list()
    with open(filename, "r", encoding="utf-8") as f:
        doc = list()
        for ligne in f:
            if len(ligne) < 2: # fin de doc
                listeDoc.append(doc)
                doc = list()
                continue
            mots = ligne.replace("\n","").split(" ")
            doc.append((mots[0], mots[1])) # mettre mots[2] Ã  la place de mots[1] pour le chuncking
    return listeDoc

%%

In [ ]:
bSmall = True

In [ ]:
if(bSmall==True):
    filename = "datasets/conll2000/chtrain.txt"
    filenameT = "datasets/conll2000/chtest.txt"
else:
    filename = "datasets/conll2000/train.txt"
    filenameT = "datasets/conll2000/test.txt"

In [ ]:
alldocs = load(filename)
alldocsT = load(filenameT)

In [ ]:
print(len(alldocs)," docs read")
print(len(alldocsT)," docs (T) read")

%%[markdown]<br>
# 1) Word embedding for classifying each word<br>
### Pre-trained word2vec

%%

In [ ]:
bload = False # Changed to false for direct download if needed
fname = "word2vec-google-news-300"

In [ ]:
if bload:
    wv_pre_trained = KeyedVectors.load(fname + ".dat")
else:
    print("Loading Pretrained Model...")
    wv_pre_trained = api.load(fname)

%% [markdown]<br>
### Some token on the dataset are missing, we will encode them with a random vector

%%

In [ ]:
def randomvec():
    default = np.random.randn(300)
    default = default / np.linalg.norm(default)
    return default

%%

In [ ]:
np.random.seed(seed=10) # seed the randomness

In [ ]:
dictadd = dict()
for d in alldocs:
    for (x, pos) in d:
        if (not (x in wv_pre_trained) and not (x in dictadd)):
            dictadd[x] = randomvec()

In [ ]:
for d in alldocsT:
    for (x, pos) in d:
        if (not (x in wv_pre_trained) and not (x in dictadd)):
            dictadd[x] = randomvec()

%% [markdown]<br>
### Store the train and test datasets: a word embedding for each token in the sequences

%%<br>
Mapping words to their embedding (either from pretrained or the random dict)

In [ ]:
wvectors = np.array([wv_pre_trained[m] if m in wv_pre_trained else dictadd[m] for d in alldocs for (m, pos) in d])
wvectorsT = np.array([wv_pre_trained[m] if m in wv_pre_trained else dictadd[m] for d in alldocsT for (m, pos) in d])

%% [markdown]<br>
### Check the size of your train/test datasets

%%

In [ ]:
print("Train vectors shape:", wvectors.shape)
print("Test vectors shape:", wvectorsT.shape)

%% [markdown]<br>
### Collecting train/test labels

%%<br>
Labels train/test

In [ ]:
buf2 = [[pos for m,pos in d ] for d in alldocs]
cles = []
[cles.extend(b) for b in buf2]
cles = np.unique(np.array(cles))
cles2ind = dict(zip(cles, range(len(cles))))
nCles = len(cles)
print(nCles," keys in the dictionary")

In [ ]:
labels  = np.array([cles2ind[pos] for d in alldocs for (m, pos) in d ])
labelsT  = np.array([cles2ind.setdefault(pos, len(cles)) for d in alldocsT for (m, pos) in d ])

In [ ]:
print(labels.shape)
print(labelsT.shape)

%% [markdown]<br>
### Train a Logistic Regression Model!<br>
**And compare performances to the baseline and sequence models (HMM/CRF) of practical 2a**

%%

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
print("Training Logistic Regression on Word Embeddings...")
clf = LogisticRegression(max_iter=1000)
clf.fit(wvectors, labels)

In [ ]:
preds = clf.predict(wvectorsT)
acc = accuracy_score(labelsT, preds)
print(f"Logistic Regression Accuracy (Word Embeddings Baseline): {acc * 100:.2f}%")

%% [markdown]<br>
# 2) Using word embedding with CRF<br>
## We will define the following features functions for CRF

%%

In [ ]:
def features_wv(sentence, index):
    word = sentence[index]
    # Handle OOV words cleanly
    v = wv_pre_trained[word] if word in wv_pre_trained else dictadd.get(word, randomvec())
    d = {'f'+str(i): v[i] for i in range(300)}
    return d

In [ ]:
def features_structural(sentence, index):
    return {
        'word': sentence[index],
        'is_first': index == 0,
        'is_last': index == len(sentence) - 1,
        'is_capitalized': sentence[index][0].upper() == sentence[index][0],
        'is_all_caps': sentence[index].upper() == sentence[index],
        'is_all_lower': sentence[index].lower() == sentence[index],
        'prefix-1': sentence[index][0],
        'prefix-2': sentence[index][:2],
        'prefix-3': sentence[index][:3],
        'suffix-1': sentence[index][-1],
        'suffix-2': sentence[index][-2:],
        'suffix-3': sentence[index][-3:],
        'prev_word': '' if index == 0 else sentence[index - 1],
        'next_word': '' if index == len(sentence) - 1 else sentence[index + 1],
        'has_hyphen': '-' in sentence[index],
        'is_numeric': sentence[index].isdigit(),
        'capitals_inside': sentence[index][1:].lower() != sentence[index][1:]
    }

In [ ]:
def features_wv_plus_structural(sentence, index):
    # Combines both dictionaries
    d = features_wv(sentence, index)
    return {**d, **features_structural(sentence, index)}

%%[markdown]<br>
## [Question]: explain what the 3 feature functions encode and what their differences are<br>
<br>
**Answer:**<br>
1. **`features_wv`**: Encodes the pure **semantic representation** of the word using the 300 dimensions of the continuous vector space (Word2Vec). It evaluates *meaning*, but ignores the context inside the sentence and morpho-syntactic cues (like capital letters).<br>
2. **`features_structural`**: Encodes **syntactic, morphological and contextual cues**. It captures suffixes/prefixes (crucial for verb tenses and POS tagging), capitalization (Crucial for Named Entities), and immediate context (`prev_word`, `next_word`). These are sparse text features.<br>
3. **`features_wv_plus_structural`**: This is a **hybrid approach** that merges the strengths of both representations : the dense semantic continuity of embeddings and the precise syntactic rules from structural features.

%% [markdown]<br>
### You can now train a CRF with the 3 features and analyse the results

%%

In [ ]:
from nltk.tag.crf import CRFTagger
import os

In [ ]:
os.makedirs('out', exist_ok=True)
test_sents = [[m for m, pos in d] for d in alldocsT]

In [ ]:
def evaluate_crf(tagger, name):
    preds = tagger.tag_sents(test_sents)
    correct, total = 0, 0
    for doc_true, doc_pred in zip(alldocsT, preds):
        for (_, true_pos), (_, pred_pos) in zip(doc_true, doc_pred):
            if true_pos == pred_pos:
                correct += 1
            total += 1
    print(f"{name} Accuracy: {correct/total*100:.2f}%")

1. Train CRF with Word Vectors

In [ ]:
print("\nTraining CRF with Word Vectors only...")
tagger_wv = CRFTagger(feature_func=features_wv)
tagger_wv.train(alldocs, 'out/crf_wv.model')
evaluate_crf(tagger_wv, "CRF (Word Vectors)")

2. Train CRF with Structural Features

In [ ]:
print("\nTraining CRF with Structural Features only...")
tagger_struct = CRFTagger(feature_func=features_structural)
tagger_struct.train(alldocs, 'out/crf_struct.model')
evaluate_crf(tagger_struct, "CRF (Structural Features)")

3. Train CRF with Combined Features

In [ ]:
print("\nTraining CRF with Combined Features...")
tagger_combined = CRFTagger(feature_func=features_wv_plus_structural)
tagger_combined.train(alldocs, 'out/crf_combined.model')
evaluate_crf(tagger_combined, "CRF (Combined Features)")